In [1]:
#importing libraries
import logging
import os
import json
import time
import uuid

from pathlib import Path
from datetime import datetime, timezone

In [2]:
#Creating the logs directory
LOG_DIR = Path("logs")

LOG_DIR.mkdir(exist_ok=True)

LOG_FILE = LOG_DIR / "app.log"

print("Log file:", LOG_FILE)

Log file: logs\app.log


In [3]:
#Configure logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    handlers=[
        logging.FileHandler(LOG_FILE),
        logging.StreamHandler()
    ],
    force=True
)

logger = logging.getLogger("rag_application")

In [4]:
#Testing logging
logger.debug("This is a debug message.")
logger.info("RAG application started.")
logger.warning("No document was found for one query.")
logger.error("An API request failed.")

2026-11-10 21:58:19,263 | INFO | RAG application started.
2026-11-10 21:58:19,268 | WARNING | No document was found for one query.
2026-11-10 21:58:19,270 | ERROR | An API request failed.


In [5]:
#Creating a JSONL logging function
JSON_LOG_FILE = LOG_DIR / "app.jsonl"

def log_event(event_name, request_id, **details):

    event = {
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "event": event_name,
        "request_id": request_id,
        **details
    }

    with open(JSON_LOG_FILE, "a", encoding="utf-8") as file:
        file.write(json.dumps(event) + "\n")

    return event

In [6]:
#Testing it
request_id = str(uuid.uuid4())

log_event(
    "request_started",
    request_id,
    question_category="RAG"
)

log_event(
    "retrieval_completed",
    request_id,
    latency_ms=180,
    retrieved_chunks=3
)

{'timestamp': '2026-11-10T16:30:01.056573+00:00',
 'event': 'retrieval_completed',
 'request_id': '4fc6b37a-6247-4d2a-a708-1ec60ff07aba',
 'latency_ms': 180,
 'retrieved_chunks': 3}

In [7]:
#Inspecting them
with open(JSON_LOG_FILE, "r", encoding="utf-8") as file:
    print(file.read())

{"timestamp": "2026-11-10T16:30:01.054647+00:00", "event": "request_started", "request_id": "4fc6b37a-6247-4d2a-a708-1ec60ff07aba", "question_category": "RAG"}
{"timestamp": "2026-11-10T16:30:01.056573+00:00", "event": "retrieval_completed", "request_id": "4fc6b37a-6247-4d2a-a708-1ec60ff07aba", "latency_ms": 180, "retrieved_chunks": 3}



In [8]:
#measuring a fucntion's execution time
def measure_latency(function):

    start_time = time.perf_counter()

    result = function()

    end_time = time.perf_counter()

    latency = end_time - start_time

    return result, latency

In [9]:
#Testing with simulated retrieval function
def retrieve_documents():

    time.sleep(0.5)

    return [
        "RAG combines retrieval and generation.",
        "Embeddings represent text as vectors.",
        "Vector databases store embeddings."
    ]

In [10]:
documents, latency = measure_latency(retrieve_documents)

print("Retrieved documents:", documents)
print("Latency:", round(latency, 3), "seconds")

Retrieved documents: ['RAG combines retrieval and generation.', 'Embeddings represent text as vectors.', 'Vector databases store embeddings.']
Latency: 0.5 seconds


In [11]:
#Measuring latency in milli seconds
latency_ms = latency * 1000

print("Latency:", round(latency_ms, 2), "ms")

Latency: 500.28 ms


In [12]:
#Create a trace operation
def trace_operation(operation_name, request_id, function):

    start_time = time.perf_counter()

    log_event(
        "operation_started",
        request_id,
        operation=operation_name
    )

    try:

        result = function()

        latency_ms = (
            time.perf_counter() - start_time
        ) * 1000

        log_event(
            "operation_completed",
            request_id,
            operation=operation_name,
            latency_ms=round(latency_ms, 2),
            status="success"
        )

        return result

    except Exception as error:

        latency_ms = (
            time.perf_counter() - start_time
        ) * 1000

        log_event(
            "operation_failed",
            request_id,
            operation=operation_name,
            latency_ms=round(latency_ms, 2),
            status="error",
            error_type=type(error).__name__
        )

        raise

In [13]:
#Simulating a RAG pipeline
def embed_question():

    time.sleep(0.1)

    return [0.12, 0.45, 0.88]


def retrieve_context():

    time.sleep(0.2)

    return [
        "RAG combines document retrieval with generation.",
        "Embeddings represent text as numerical vectors.",
        "Vector databases support similarity search."
    ]


def generate_answer():

    time.sleep(0.4)

    return "RAG retrieves relevant information and uses it to generate an answer."

In [14]:
def run_observed_rag():

    request_id = str(uuid.uuid4())

    total_start = time.perf_counter()

    log_event(
        "request_started",
        request_id
    )

    try:

        embedding = trace_operation(
            "embedding",
            request_id,
            embed_question
        )

        context = trace_operation(
            "retrieval",
            request_id,
            retrieve_context
        )

        answer = trace_operation(
            "generation",
            request_id,
            generate_answer
        )

        total_latency_ms = (
            time.perf_counter() - total_start
        ) * 1000

        log_event(
            "request_completed",
            request_id,
            total_latency_ms=round(total_latency_ms, 2),
            status="success"
        )

        return {
            "request_id": request_id,
            "answer": answer,
            "retrieved_chunks": len(context),
            "total_latency_ms": round(total_latency_ms, 2)
        }

    except Exception:

        log_event(
            "request_failed",
            request_id,
            status="error"
        )

        raise

In [15]:
result = run_observed_rag()

print(json.dumps(result, indent=4))

{
    "request_id": "79c49519-b0c0-4d8d-8f48-37fc57945f41",
    "answer": "RAG retrieves relevant information and uses it to generate an answer.",
    "retrieved_chunks": 3,
    "total_latency_ms": 710.92
}
